# Do Causal Lead-Lag Relationships in ETF Returns Survive Out of Sample?

**Author:** Stephen Maranatha Asiedu

## Summary
Financial returns are a multivariate time series in which one market might *lead* another: bond
moves preceding equity sector moves, large caps leading small caps, and so on. Causal discovery
methods for time series claim to recover this directed lead-lag structure. This notebook asks the
practical question a trading desk would ask: **does the discovered structure forecast next-day
returns out of sample, does the structure stay stable over time, and does any edge survive
transaction costs?**

**Data.** Daily adjusted closes for 14 liquid US ETFs (equity indices, sectors, Treasuries, gold,
high-yield credit), 2010–2025, from Yahoo Finance.

**Time-series pipeline.**
1. Stationarity (ADF tests on prices vs log returns), autocorrelation and volatility clustering
   (ACF of returns vs squared returns, Ljung–Box tests), heavy tails, rolling volatility and
   rolling cross-correlation.
2. Volatility standardisation with a lagged EWMA estimate (no look-ahead), because heteroskedasticity
   breaks the F-tests used for Granger causality.
3. Three forecasting models, refit in a rolling **walk-forward** scheme (2-year training window,
   monthly refits):
   - **AR**: each asset's own lags (univariate baseline)
   - **VAR**: ridge regression on lags of every asset (dense multivariate baseline)
   - **CAUSAL**: parents chosen by *conditional* Granger F-tests with Benjamini–Hochberg FDR
     control on the training window only, then OLS on those parents (sparse causal model)
4. Evaluation: out-of-sample R², Diebold–Mariano tests with Newey–West errors, a sign-trading
   strategy's Sharpe ratio net of costs, stability of the causal graph across windows, and results
   split by market regime (COVID crash, 2022 rate shock).
5. A **synthetic control** with a known lead-lag graph, heavy-tailed shocks and volatility regimes,
   so the method's ability to recover structure is measured before it is trusted on real data.

Open in Colab and choose *Runtime → Run all*. The findings section at the end is generated from
the computed results.

In [ ]:
!pip -q install yfinance statsmodels

In [ ]:
import warnings, numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
from statsmodels.tsa.stattools import adfuller, acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from IPython.display import display, Markdown
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

# ---------------- configuration
USE_SYNTHETIC_ONLY = False         # True = skip real data (for offline testing)
TICKERS = ["SPY","QQQ","IWM","XLK","XLF","XLE","XLV","XLY","XLP","XLI","XLU","TLT","GLD","HYG"]
START, END = "2010-01-01", "2025-12-31"
P = 2            # number of lags
TRAIN = 504      # training window (~2 years of trading days)
STEP = 21        # refit every ~month
FDR_Q = 0.05     # Benjamini-Hochberg level for Granger edges
RIDGE = 0.05     # ridge penalty for the VAR baseline
COST_BPS = 1.0   # one-way transaction cost
EWMA_HALFLIFE = 20

## 1. Data
Log returns $r_t = \log P_t - \log P_{t-1}$ are used throughout. Prices are adjusted for splits and
dividends.

In [ ]:
def simulate(n=12, T=4000, p=2, density=0.08, seed=0):
    # Sparse VAR(p) with Student-t(4) shocks, correlated noise and a periodic high-vol regime.
    rng = np.random.default_rng(seed)
    A = np.zeros((p, n, n)); truth = np.zeros((n, n), bool)   # truth[target, source]
    for j in range(n):
        for i in range(n):
            if i != j and rng.random() < density:
                truth[j, i] = True
                A[rng.integers(p), j, i] = rng.choice([-1, 1]) * rng.uniform(0.08, 0.2)
    comp = np.zeros((n*p, n*p)); comp[:n] = np.hstack(A); comp[n:, :-n] = np.eye(n*(p-1))
    rad = np.abs(np.linalg.eigvals(comp)).max()
    if rad > 0.9: A *= 0.9 / rad
    L = np.linalg.cholesky(0.3*np.ones((n, n)) + 0.7*np.eye(n))
    vol = np.where((np.arange(T)//500) % 4 == 3, 2.5, 1.0)
    X = np.zeros((T, n))
    for t in range(p, T):
        X[t] = sum(A[k] @ X[t-1-k] for k in range(p)) + L @ (rng.standard_t(4, n)/np.sqrt(2)) * 0.01 * vol[t]
    idx = pd.bdate_range("2010-01-04", periods=T-p)
    return pd.DataFrame(X[p:], index=idx, columns=[f"A{i}" for i in range(n)]), truth

if USE_SYNTHETIC_ONLY:
    rets, _ = simulate(n=len(TICKERS), seed=1)
    rets.columns = TICKERS
    prices = np.exp(rets.cumsum()) * 100
    print("Offline mode: 'real' data replaced by a simulation.")
else:
    import yfinance as yf
    prices = yf.download(TICKERS, start=START, end=END, auto_adjust=True, progress=False)["Close"][TICKERS].dropna()
    rets = np.log(prices).diff().dropna()
print(rets.shape, rets.index.min().date(), "->", rets.index.max().date())
rets.describe().T[["mean", "std", "min", "max"]].mul(100).round(3).rename(columns=lambda c: c + " (%)")

## 2. Time-series diagnostics
Before modelling, check the stylised facts that drive every modelling choice later:

- **Stationarity.** Prices should have a unit root; returns should not. Regressions on
  non-stationary series produce spurious results.
- **Weak linear autocorrelation in returns** (markets are close to efficient) but **strong
  autocorrelation in squared returns** (volatility clustering, i.e. conditional heteroskedasticity).
- **Heavy tails** (excess kurtosis), which matter because Granger F-tests assume roughly Gaussian errors.

In [ ]:
rows = []
for t in TICKERS:
    lp = np.log(prices[t]); r = rets[t]
    lb_r  = acorr_ljungbox(r, lags=[10])["lb_pvalue"].iloc[0]
    lb_r2 = acorr_ljungbox(r**2, lags=[10])["lb_pvalue"].iloc[0]
    rows.append({"ticker": t,
                 "ADF p (log price)": adfuller(lp, autolag="AIC")[1],
                 "ADF p (return)": adfuller(r, autolag="AIC")[1],
                 "LB(10) p returns": lb_r, "LB(10) p squared": lb_r2,
                 "excess kurtosis": stats.kurtosis(r), "skew": stats.skew(r)})
diag = pd.DataFrame(rows).set_index("ticker")
display(diag.round(4))

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13, 8))
(prices / prices.iloc[0]).plot(ax=ax[0, 0], legend=False, lw=0.8, logy=True)
ax[0, 0].set_title("Normalised prices (log scale): non-stationary")
rets["SPY"].plot(ax=ax[0, 1], lw=0.5, color="k")
ax[0, 1].set_title("SPY daily log returns: stationary, but volatility clusters")
lags = range(1, 31)
ax[1, 0].bar(lags, acf(rets["SPY"], nlags=30)[1:], color="C0")
ax[1, 0].axhline(1.96/np.sqrt(len(rets)), ls="--", c="r"); ax[1, 0].axhline(-1.96/np.sqrt(len(rets)), ls="--", c="r")
ax[1, 0].set_title("ACF of SPY returns (near zero)")
ax[1, 1].bar(lags, acf(rets["SPY"]**2, nlags=30)[1:], color="C3")
ax[1, 1].axhline(1.96/np.sqrt(len(rets)), ls="--", c="r")
ax[1, 1].set_title("ACF of squared SPY returns (persistent: volatility clustering)")
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
(rets[["SPY", "TLT", "GLD", "HYG"]].rolling(63).std() * np.sqrt(252) * 100).plot(ax=ax[0], lw=0.9)
ax[0].set_title("Rolling 3-month annualised volatility (%)")
roll_corr = rets[["TLT", "GLD", "HYG", "IWM"]].rolling(126).corr(rets["SPY"])
roll_corr.plot(ax=ax[1], lw=0.9)
ax[1].axhline(0, c="k", lw=0.6); ax[1].set_title("Rolling 6-month correlation with SPY")
plt.tight_layout(); plt.show()

**Volatility standardisation.** Returns are divided by an EWMA volatility estimate computed from
data up to $t-1$ only:
$$z_t = r_t / \hat\sigma_{t|t-1}, \qquad \hat\sigma^2_{t|t-1} = \text{EWMA}(r^2)_{t-1}.$$
This removes most of the heteroskedasticity so Granger tests are not dominated by crisis periods.
Models are fitted on both raw and standardised returns.

In [ ]:
def ewma_scale(r, halflife=EWMA_HALFLIFE):
    sig = np.sqrt((r**2).ewm(halflife=halflife, min_periods=60).mean().shift(1))
    return (r / sig).dropna()

z = ewma_scale(rets)
r_al = rets.loc[z.index]
cmp = pd.DataFrame({"kurtosis raw": rets.apply(stats.kurtosis),
                    "kurtosis scaled": z.apply(stats.kurtosis),
                    "LB(10) p sq raw": [acorr_ljungbox(rets[t]**2, lags=[10])["lb_pvalue"].iloc[0] for t in TICKERS],
                    "LB(10) p sq scaled": [acorr_ljungbox(z[t]**2, lags=[10])["lb_pvalue"].iloc[0] for t in TICKERS]},
                   index=TICKERS)
display(cmp.round(4))

## 3. Models
For target asset $j$ with $p$ lags, the full model is
$$r_{j,t} = c + \sum_{k=1}^{p}\sum_{i} a^{(k)}_{ji}\, r_{i,t-k} + \varepsilon_t.$$

**Conditional Granger causality.** Source $i$ Granger-causes $j$ if dropping all $p$ lags of $i$
from the *full* model (which conditions on every other asset's past) significantly increases the
residual sum of squares, tested with an $F(p,\, T-np-1)$ statistic. Conditioning on all other
assets is what separates a direct lead-lag link from one that is driven by a common factor. The
$n-1$ p-values per target are corrected with Benjamini–Hochberg, so the expected fraction of false
edges is controlled at $q$.

All selection and fitting use the training window only; forecasts are made for the next 21 days and
the window then rolls forward.

In [ ]:
def lag_matrix(R, p):
    T, n = R.shape
    return np.hstack([R[p-k-1:T-k-1] for k in range(p)]), R[p:]

def ols(Z, y):
    Z1 = np.column_stack([np.ones(len(Z)), Z])
    b, *_ = np.linalg.lstsq(Z1, y, rcond=None)
    e = y - Z1 @ b
    return b, e @ e

def ridge(Z, y, lam):
    mu, sd = Z.mean(0), Z.std(0) + 1e-12
    Zs = (Z - mu) / sd
    b = np.linalg.solve(Zs.T @ Zs + lam*len(Zs)*np.eye(Zs.shape[1]), Zs.T @ (y - y.mean()))
    return lambda Zn: y.mean() + ((Zn - mu) / sd) @ b

cols_for = lambda src, n, p: [k*n + i for k in range(p) for i in src]

def granger_parents(Z, y, j, n, p, q):
    _, rss_f = ols(Z, y); df = len(y) - Z.shape[1] - 1
    cand = [i for i in range(n) if i != j]; pv = []
    for i in cand:
        drop = set(cols_for([i], n, p))
        _, rss_r = ols(Z[:, [c for c in range(Z.shape[1]) if c not in drop]], y)
        pv.append(stats.f.sf(((rss_r - rss_f)/p) / (rss_f/df), p, df))
    pv = np.array(pv); o = np.argsort(pv); m = len(pv)
    ok = pv[o] <= q*np.arange(1, m+1)/m
    k = np.max(np.where(ok)[0]) + 1 if ok.any() else 0
    return sorted(cand[x] for x in o[:k])

def walk_forward(df, p=P, train=TRAIN, step=STEP, lam=RIDGE, q=FDR_Q):
    Z, Y = lag_matrix(df.values, p); T, n = Y.shape
    idx = df.index[p:]
    preds = {m: np.full((T, n), np.nan) for m in ["AR", "VAR", "CAUSAL"]}
    graphs, gdates = [], []
    for s in range(train, T, step):
        tr, te = slice(s-train, s), slice(s, min(s+step, T))
        G = np.zeros((n, n), bool)
        for j in range(n):
            y = Y[tr, j]; own = cols_for([j], n, p)
            b, _ = ols(Z[tr][:, own], y); preds["AR"][te, j] = b[0] + Z[te][:, own] @ b[1:]
            preds["VAR"][te, j] = ridge(Z[tr], y, lam)(Z[te])
            par = granger_parents(Z[tr], y, j, n, p, q); G[j, par] = True
            c = cols_for([j]+par, n, p)
            b, _ = ols(Z[tr][:, c], y); preds["CAUSAL"][te, j] = b[0] + Z[te][:, c] @ b[1:]
        graphs.append(G); gdates.append(idx[s])
    preds = {m: pd.DataFrame(v, index=idx, columns=df.columns) for m, v in preds.items()}
    return pd.DataFrame(Y, index=idx, columns=df.columns), preds, np.array(graphs), pd.DatetimeIndex(gdates)

In [ ]:
def oos_r2(y, f):
    m = f.notna(); return 1 - ((y[m]-f[m])**2).sum() / (y[m]**2).sum()   # benchmark: zero forecast

def strategy(raw, F, cost_bps=COST_BPS):
    F = F.dropna(); R = raw.loc[F.index]
    pos = np.sign(F) / F.shape[1]
    gross = (pos * R).sum(1)
    turn = pos.diff().abs().sum(1).fillna(pos.abs().sum(1))
    return gross, gross - turn*cost_bps/1e4, turn

sharpe = lambda x: np.sqrt(252) * x.mean() / x.std()

def dm_test(y, f1, f2, lag=5):
    # Diebold-Mariano on squared-error loss pooled across assets; positive stat = f2 better.
    m = f1.notna().all(1) & f2.notna().all(1)
    d = (((y[m]-f1[m])**2).mean(1) - ((y[m]-f2[m])**2).mean(1)).values
    T = len(d); dbar = d.mean(); u = d - dbar
    lrv = u @ u / T + 2*sum((1-k/(lag+1)) * (u[k:] @ u[:-k]) / T for k in range(1, lag+1))
    stat = dbar / np.sqrt(lrv / T)
    return stat, 2*stats.norm.sf(abs(stat))

def jaccard_series(graphs):
    out = []
    for a, b in zip(graphs[:-1], graphs[1:]):
        u = (a | b).sum(); out.append((a & b).sum()/u if u else np.nan)   # skip pairs of empty graphs
    return np.array(out)

def evaluate(Y, preds, raw):
    rows = []
    for m, F in preds.items():
        r2 = pd.Series({c: oos_r2(Y[c], F[c]) for c in Y.columns})
        g, n_, t = strategy(raw, F)
        rows.append({"model": m, "median OOS R2 (%)": 100*r2.median(), "assets R2>0": int((r2 > 0).sum()),
                     "Sharpe gross": sharpe(g), "Sharpe net": sharpe(n_), "daily turnover": t.mean()})
    return pd.DataFrame(rows).set_index("model")

## 4. Synthetic control: can the method recover a known lead-lag graph?
A sparse VAR(2) with 14 series, about 8% of possible edges active, Student-t(4) shocks, correlated
noise and a stress regime with 2.5× volatility every fourth block. This mimics the difficulty of
real returns while keeping the true graph known. An edge counts as "discovered" if it is selected
in more than half of the walk-forward windows.

In [ ]:
syn, truth = simulate(n=len(TICKERS), seed=0)
syn_res = {}
for label, data in [("raw", syn), ("vol-scaled", ewma_scale(syn))]:
    Ys, ps, gs, _ = walk_forward(data)
    est = gs.mean(0) > 0.5; tp = (est & truth).sum()
    syn_res[label] = {"precision": tp/max(est.sum(), 1), "recall": tp/truth.sum(),
                      "true edges": int(truth.sum()), "found edges": int(est.sum()),
                      "graph stability (Jaccard)": np.nanmean(jaccard_series(gs)),
                      "median OOS R2 CAUSAL (%)": 100*np.median([oos_r2(Ys[c], ps["CAUSAL"][c]) for c in Ys.columns]),
                      "median OOS R2 AR (%)": 100*np.median([oos_r2(Ys[c], ps["AR"][c]) for c in Ys.columns])}
syn_tab = pd.DataFrame(syn_res).T
display(syn_tab.round(3))

## 5. Real data: walk-forward results
Models are fitted on raw returns and on EWMA-standardised returns. Strategy P&L always uses raw
returns, trading the sign of each forecast with equal weight across assets.

In [ ]:
Y_raw, P_raw, G_raw, D_raw = walk_forward(rets)
Y_z,   P_z,   G_z,   D_z   = walk_forward(z)
res_raw = evaluate(Y_raw, P_raw, rets); res_z = evaluate(Y_z, P_z, r_al)
display(Markdown("**Raw returns**")); display(res_raw.round(3))
display(Markdown("**EWMA vol-standardised returns**")); display(res_z.round(3))

dm = {}
for lab, Y_, P_ in [("raw", Y_raw, P_raw), ("vol-scaled", Y_z, P_z)]:
    for other in ["AR", "VAR"]:
        s, pval = dm_test(Y_, P_[other], P_["CAUSAL"]); dm[(lab, f"CAUSAL vs {other}")] = {"DM stat": s, "p-value": pval}
dm = pd.DataFrame(dm).T
display(Markdown("**Diebold–Mariano tests** (positive = CAUSAL has lower squared error)")); display(dm.round(3))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
for m in P_z:
    g, n_, _ = strategy(r_al, P_z[m]); n_.cumsum().mul(100).plot(ax=ax[0], label=f"{m} (net)")
ax[0].set_title(f"Cumulative net P&L of sign strategy, vol-scaled models, {COST_BPS} bp cost (%)"); ax[0].legend()

def rolling_r2(Y, F, w=252):
    m = F.notna().all(1); e = ((Y[m]-F[m])**2).sum(1); s = (Y[m]**2).sum(1)
    return 1 - e.rolling(w).sum() / s.rolling(w).sum()
for m in P_z: rolling_r2(Y_z, P_z[m]).mul(100).plot(ax=ax[1], label=m, lw=1)
ax[1].axhline(0, c="k", lw=0.7); ax[1].set_title("Rolling 1-year out-of-sample R² (%)"); ax[1].legend()
plt.tight_layout(); plt.show()

## 6. Is the causal graph stable through time?
A real lead-lag relationship should be selected in most windows. If edges flicker on and off, the
"structure" is mostly estimation noise or a regime-specific artefact.

In [ ]:
freq = pd.DataFrame(G_z.mean(0), index=TICKERS, columns=TICKERS)   # row = target, column = source
fig, ax = plt.subplots(1, 2, figsize=(14, 5.5), gridspec_kw={"width_ratios": [1, 1.3]})
im = ax[0].imshow(freq.values, cmap="viridis", vmin=0, vmax=max(0.5, freq.values.max()))
ax[0].set_xticks(range(len(TICKERS)), TICKERS, rotation=90); ax[0].set_yticks(range(len(TICKERS)), TICKERS)
ax[0].set_xlabel("source (leads)"); ax[0].set_ylabel("target (lags)")
ax[0].set_title("Share of windows each edge is selected"); fig.colorbar(im, ax=ax[0], shrink=0.8)
js = pd.Series(jaccard_series(G_z), index=D_z[1:])
ne = pd.Series(G_z.sum((1, 2)), index=D_z)
ax[1].plot(ne.index, ne.values, label="edges selected", color="C0")
ax2 = ax[1].twinx(); ax2.plot(js.index, js.rolling(6, min_periods=1).mean(), color="C3", label="Jaccard (6-window mean)")
ax[1].set_title("Graph size and window-to-window stability"); ax[1].set_ylabel("edges"); ax2.set_ylabel("Jaccard")
ax[1].legend(loc="upper left"); ax2.legend(loc="upper right")
plt.tight_layout(); plt.show()

top = (freq.stack().rename("share").reset_index().rename(columns={"level_0": "target", "level_1": "source"})
       .query("target != source").sort_values("share", ascending=False).head(10))
display(top[["source", "target", "share"]].assign(edge=lambda d: d.source + " → " + d.target).set_index("edge")[["share"]].round(3))

## 7. Regimes and transaction costs
Lead-lag effects may only appear in stressed markets, when liquidity is thin and information
diffuses more slowly. Results are split into calm and stress periods, and the net Sharpe is
re-computed across a range of costs.

In [ ]:
REGIMES = {"2012–2019": ("2012-01-01", "2019-12-31"), "COVID (Feb–Jun 2020)": ("2020-02-15", "2020-06-30"),
           "2022 rate shock": ("2022-01-01", "2022-12-31"), "2023–2025": ("2023-01-01", "2025-12-31")}
reg = {}
for name, (a, b) in REGIMES.items():
    for m in ["AR", "CAUSAL"]:
        F = P_z[m].loc[a:b].dropna(); Yr = Y_z.loc[F.index]
        if len(F) < 20: continue
        _, n_, _ = strategy(r_al, F)
        reg[(name, m)] = {"days": len(F), "median OOS R2 (%)": 100*np.median([oos_r2(Yr[c], F[c]) for c in TICKERS]),
                          "Sharpe net": sharpe(n_)}
reg = pd.DataFrame(reg).T; display(reg.round(3))

costs = [0, 0.5, 1, 2, 5]
cs = pd.DataFrame({m: [sharpe(strategy(r_al, P_z[m], c)[1]) for c in costs] for m in P_z}, index=costs)
cs.index.name = "cost (bp)"; display(cs.round(3))

## 8. Findings (generated from the results above)

In [ ]:
cz = res_z.loc["CAUSAL"]; az = res_z.loc["AR"]; vz = res_z.loc["VAR"]
dmc = dm.loc[("vol-scaled", "CAUSAL vs AR")]
sv = syn_tab.loc["vol-scaled"]
stab = np.nanmean(jaccard_series(G_z)); n_empty = int((G_z.sum((1,2)) == 0).sum())
best = top.iloc[0]
be = next((c for c in costs if cs.loc[c, "CAUSAL"] <= 0), None)
dmp = dmc["p-value"]
lines = [
 f"**Data.** {len(rets):,} trading days of 14 ETFs; {len(D_z)} walk-forward refits of a {TRAIN}-day window.",
 f"**Stylised facts.** Log prices are non-stationary (median ADF p = {diag['ADF p (log price)'].median():.2f}) and returns are stationary "
 f"(max ADF p = {diag['ADF p (return)'].max():.3f}). Squared returns are strongly autocorrelated in "
 f"{(diag['LB(10) p squared'] < 0.01).sum()}/14 assets (volatility clustering); median excess kurtosis is {diag['excess kurtosis'].median():.1f}. "
 f"EWMA standardisation cuts median kurtosis to {cmp['kurtosis scaled'].median():.1f}.",
 f"**Synthetic control.** On a simulated VAR with {int(sv['true edges'])} true edges, the causal selector found {int(sv['found edges'])} stable edges "
 f"with precision {sv['precision']:.2f} and recall {sv['recall']:.2f}: what it finds is usually real, but weak links are missed with 2-year windows and fat tails.",
 f"**Out-of-sample forecasting (vol-scaled).** Median OOS R² is {cz['median OOS R2 (%)']:.2f}% for CAUSAL, {az['median OOS R2 (%)']:.2f}% for AR and "
 f"{vz['median OOS R2 (%)']:.2f}% for the dense VAR; {int(cz['assets R2>0'])}/14 assets have positive R² under CAUSAL. "
 f"Diebold–Mariano CAUSAL vs AR: stat {dmc['DM stat']:.2f}, p = {dmp:.3f} "
 + ("(not significant at 5%)." if dmp >= 0.05 else
    ("— CAUSAL is significantly MORE accurate than AR." if dmc['DM stat'] > 0 else
     "— CAUSAL is significantly LESS accurate than AR.")),
 f"**Graph stability.** On average {G_z.sum((1,2)).mean():.1f} edges are selected per window, with mean window-to-window Jaccard similarity {stab:.2f}. "
 f"{n_empty}/{len(G_z)} windows select no edges at all. "
 f"The most persistent edge is {best.source} → {best.target}, selected in {best.share:.0%} of windows.",
 f"**Trading.** Sign-strategy Sharpe for CAUSAL is {cz['Sharpe gross']:.2f} gross and {cz['Sharpe net']:.2f} net of {COST_BPS} bp, "
 f"with mean daily turnover {cz['daily turnover']:.2f}. " + (f"Net Sharpe turns non-positive by {be} bp." if be is not None else "Net Sharpe stays positive up to 5 bp."),
 "**Regimes.** " + "; ".join(f"{r}: CAUSAL net Sharpe {reg.loc[(r,'CAUSAL'),'Sharpe net']:.2f} vs AR {reg.loc[(r,'AR'),'Sharpe net']:.2f}"
                              for r in REGIMES if (r, "CAUSAL") in reg.index) + ".",
]
display(Markdown("\n\n".join("- " + l for l in lines)))

## 9. Discussion and limitations
- **Interpretation.** Write your conclusion here after reading Section 8: whether sparse causal
  structure beat the baselines, whether any edge was persistent, and whether the result survives costs.
- **Why it is hard.** Daily lead-lag between liquid ETFs is heavily arbitraged; effects are small
  relative to noise, heavy tails and changing volatility, and the synthetic control shows how much
  power a 2-year window has.
- **Limitations.** Linear models only; close-to-close data ignores asynchronous closes (GLD and TLT
  trade on different venues); Yahoo data is not survivorship- or error-checked; equal-weight sign
  trading is a deliberately crude test of economic value.
- **Extensions.** Intraday data, PCMCI-style conditional independence tests in place of linear
  Granger, regime-switching VARs, and weekly horizons.